<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [10]</a>'.</span>

# 09 - Anomaly Detection

**The deliverable the brief asked for and the original project never built.**

There is no `IsolationForest`, no `LocalOutlierFactor` and no one-class model anywhere in
the original repo; the closest thing was DBSCAN labelling 130 points as noise inside the
clustering notebook. This notebook closes that gap - and it happens to be the most
commercially useful model in the project.

Four products fall out of one model family:

| Section | What it does |
|---|---|
| 1. Scam / bait detection | Outliers in the **joint** (specs, price) distribution |
| 2. Underpriced-deal finder | The residual, flipped: `predicted >> actual` |
| 3. Spec-inconsistency detection | Listings that violate high-confidence association rules |
| 4. Evaluation | Detector agreement, and precision@k against hand labels |

Section 3 is the bridge that makes regression, association rules and anomaly detection read
as one project instead of three separate assignments.

In [1]:
# --- project bootstrap ---
import os, sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError("pyproject.toml not found - run this notebook inside the repo")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"project root: {PROJECT_ROOT}")


project root: /app


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from laptop_price.anomaly.detectors import fit_detectors, score_listings, detector_agreement
from laptop_price.anomaly.deals import classify_listings, rank_deals, residual_scores
from laptop_price.anomaly.rules_check import load_rules, check_spec_consistency, summarise_violations
from laptop_price.data import load_model_ready
from laptop_price.models.registry import load_bundle
from laptop_price.features.build import TARGET

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

matrix = load_model_ready()
bundle = load_bundle()
print(f"{len(matrix):,} listings | model {bundle.version}")

16,255 listings | model v20260911T0203


## 1. Scam / bait detection

An RTX 4090 machine listed at 60,000 DZD is not a bargain, it is a scam or a typo.
Neither the specs alone nor the price alone is unusual there - only the **combination** is,
which is why the detectors are fitted on the joint distribution with log-price included as
a dimension.

Four detectors rather than one. Unsupervised anomaly detection is genuinely hard to
evaluate, and agreement between independent methods is the cheapest evidence available.

In [3]:
ensemble = fit_detectors(matrix)
scores = score_listings(ensemble, matrix)

print("flagged by each detector:")
for column in [c for c in scores.columns if c.endswith("_outlier")]:
    print(f"  {column:28s} {scores[column].sum():5,d}  ({scores[column].mean():.2%})")
print(f"\nmajority vote: {scores['is_outlier'].sum():,} listings ({scores['is_outlier'].mean():.2%})")

flagged by each detector:
  isolation_forest_outlier       326  (2.01%)
  local_outlier_factor_outlier   326  (2.01%)
  ecod_outlier                     0  (0.00%)
  copod_outlier                    0  (0.00%)
  is_outlier                       2  (0.01%)

majority vote: 2 listings (0.01%)


In [4]:
agreement = detector_agreement(scores)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(agreement.astype(float), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(agreement)), [c.replace("_outlier", "") for c in agreement.columns], rotation=45, ha="right")
ax.set_yticks(range(len(agreement)), [c.replace("_outlier", "") for c in agreement.index])
for i in range(len(agreement)):
    for j in range(len(agreement)):
        ax.text(j, i, f"{agreement.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Pairwise Jaccard agreement between detectors")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

**Reading the matrix.** High off-diagonal agreement means the detectors are finding the
same structure and the flags are trustworthy. Low agreement means they are each picking up
a different artefact, and a majority vote is doing real work rather than rubber-stamping
one detector's opinion.

In [5]:
# What does an outlier actually look like?
flagged = matrix[scores["is_outlier"]].copy()
flagged["price_per_1k_cpumark"] = flagged[TARGET] / (flagged["cpu_mark"] / 1000).replace(0, np.nan)

cols = [TARGET, "RAM_SIZE", "SSD_SIZE", "cpu_mark", "gpu_g3d_mark", "brand", "price_per_1k_cpumark"]
print("cheapest flagged listings relative to their performance:")
flagged.nsmallest(10, "price_per_1k_cpumark")[cols]

cheapest flagged listings relative to their performance:


,price_corrected,RAM_SIZE,SSD_SIZE,cpu_mark,gpu_g3d_mark,brand,price_per_1k_cpumark
840,295000.0,32.0,10000.0,18492.0,5967.0,XPS,15952.844473
16107,120000.0,4.0,0.0,NaN,NaN,MAC,NaN


## 2. The underpriced-deal finder

The useful half, and it is one sign flip away from the regression model that already exists.

`residual = predicted - actual`:

* strongly positive **and otherwise ordinary** -> a genuine bargain
* strongly positive **and also a joint outlier** -> far more likely a scam or a typo
* strongly negative -> overpriced

Separating the first two is the whole trick. A deal feed that cannot tell them apart is
worse than no deal feed.

In [6]:
verdicts = classify_listings(matrix, bundle, outlier_flags=scores["is_outlier"])
print(verdicts["verdict"].value_counts().to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(verdicts["discount_pct"].clip(-150, 100), bins=80, color="#3498db", edgecolor="none")
ax.axvline(0, color="black", lw=1)
ax.axvline(35, color="#2ecc71", lw=2, ls="--", label="bargain threshold")
ax.axvline(-35, color="#e74c3c", lw=2, ls="--", label="overpriced threshold")
ax.set_xlabel("(predicted - actual) / predicted  [%]")
ax.set_ylabel("listings")
ax.set_title("How far each listing sits from the model's estimate")
ax.legend(); plt.tight_layout(); plt.show()

verdict
normal        15011
overpriced      877
bargain         262
suspicious      105


In [7]:
deals = rank_deals(matrix, bundle, top_k=20)
print("Top candidate bargains (scam-shaped listings already filtered out):")
deals[[TARGET, "predicted", "discount_pct", "brand", "city_grouped", "RAM_SIZE", "SSD_SIZE", "cpu_mark"]]

Top candidate bargains (scam-shaped listings already filtered out):


,price_corrected,predicted,discount_pct,brand,city_grouped,RAM_SIZE,SSD_SIZE,cpu_mark
0,12000.0,29918.635612,59.891219,DYNABOOK,DRARIA,6.0,120.0,2019.0
1,26500.0,65248.844119,59.386254,OTHER,EZZOUAR,8.0,240.0,12947.0
2,28000.0,68290.462573,58.998667,INSPIRON,EL BIAR,4.0,512.0,14479.0
3,10000.0,24350.746510,58.933497,OTHER,OTHER,14.0,64.0,894.0
4,25000.0,60654.290771,58.782801,MAC,KOUBA,8.0,128.0,NaN
5,45000.0,106157.675584,57.610225,PRECISION,BLIDA,8.0,256.0,28377.0
6,60000.0,140944.924502,57.430180,MAC,OTHER,16.0,512.0,19108.0
7,45000.0,105459.840022,57.329729,MACBOOK,BIRKHADEM,8.0,256.0,14133.0
8,17000.0,39070.389301,56.488788,PAVILION,SIDI BEL ABBES,128.0,32.0,887.0
9,10000.0,22972.255710,56.469229,INSPIRON,TIZI OUZOU,4.0,0.0,2633.0


This table is the product. Served daily it is a *"best deals on Ouedkniss right now"*
feed, and it is roughly forty lines on top of a model that already existed. It is exposed
as `GET /deals` in the API.

## 3. Spec-inconsistency detection

If `{RTX 4060} -> {16GB RAM}` holds at high confidence across the market, then a listing
claiming *RTX 4060 + 4GB DDR3* is probably mistyped or fraudulent.

This consumes the rules mined in notebook 08 rather than re-deriving them, which is what
makes the association-rules deliverable feed into this one.

In [8]:
rules = load_rules()
if rules.empty:
    print("No rules found in reports/rules/.")
    print("Run notebooks/08_association_rules.ipynb first, then re-run this section.")
else:
    print(f"{len(rules):,} high-confidence rules loaded")
    display(rules.nlargest(10, "confidence")[["antecedents", "consequents", "support", "confidence", "lift"]])

No rules found in reports/rules/.
Run notebooks/08_association_rules.ipynb first, then re-run this section.


In [9]:
# Build a small one-hot transaction table over the discretised specs, in the same
# vocabulary the rules were mined in.
def build_transactions(df: pd.DataFrame) -> pd.DataFrame:
    items = pd.DataFrame(index=df.index)
    items["RAM_low"]      = df["RAM_SIZE"] <= 4
    items["RAM_mid"]      = df["RAM_SIZE"].between(6, 8)
    items["RAM_high"]     = df["RAM_SIZE"] >= 16
    items["SSD_none"]     = df["SSD_SIZE"] == 0
    items["SSD_large"]    = df["SSD_SIZE"] >= 512
    items["GPU_none"]     = df["gpu_g3d_mark"].fillna(0) < 1500
    items["GPU_strong"]   = df["gpu_g3d_mark"].fillna(0) >= 10000
    items["CPU_weak"]     = df["cpu_mark"] < 5000
    items["CPU_strong"]   = df["cpu_mark"] >= 20000
    items["screen_4k"]    = df["SCREEN_RESOLUTION_ENC"] >= 8
    return items.fillna(False)

transactions = build_transactions(matrix)
print(f"{transactions.shape[1]} items, {len(transactions):,} transactions")
transactions.mean().sort_values(ascending=False).to_frame("support")

10 items, 16,255 transactions


,support
GPU_none,0.551215
RAM_high,0.550292
SSD_large,0.485758
RAM_mid,0.364319
CPU_weak,0.171516
GPU_strong,0.154906
CPU_strong,0.131713
RAM_low,0.070132
SSD_none,0.063857
screen_4k,0.008674


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [10]:
# Mine rules over this vocabulary directly, so the check works even before
# notebook 08 has been run.
from mlxtend.frequent_patterns import apriori, association_rules

frequent = apriori(transactions, min_support=0.03, use_colnames=True)
local_rules = association_rules(frequent, metric="confidence", min_threshold=0.80)
local_rules = local_rules[local_rules["lift"] > 1.1]
print(f"{len(local_rules)} rules at confidence >= 0.80, lift > 1.1")
local_rules.nlargest(10, "confidence")[["antecedents", "consequents", "support", "confidence", "lift"]]

TypeError: association_rules() missing 1 required positional argument: 'num_itemsets'

In [ ]:
consistency = check_spec_consistency(transactions, local_rules)
print(f"listings violating at least one high-confidence rule: "
      f"{consistency['is_inconsistent'].sum():,} ({consistency['is_inconsistent'].mean():.2%})")
print("\nmost frequently violated rules:")
print(summarise_violations(consistency).to_string())

In [ ]:
# A listing that is BOTH a joint outlier and spec-inconsistent is the strongest
# fraud signal the project can produce.
combined = pd.DataFrame({
    "joint_outlier": scores["is_outlier"],
    "spec_inconsistent": consistency["is_inconsistent"],
    "verdict": verdicts["verdict"],
})
print(pd.crosstab(combined["joint_outlier"], combined["spec_inconsistent"]).to_string())

both = matrix[combined["joint_outlier"] & combined["spec_inconsistent"]]
print(f"\n{len(both):,} listings flagged by both signals:")
both[[TARGET, "RAM_SIZE", "SSD_SIZE", "cpu_mark", "gpu_g3d_mark", "brand", "city_grouped"]].head(15)

## 4. Evaluation

Unsupervised anomaly detection is unfalsifiable without some ground truth. Two honest
things can be reported:

1. **Detector agreement** (section 1) - cheap, and available immediately.
2. **Precision@k against hand labels** - label ~200 listings as `scam` / `normal` /
   `bargain` into `data/labels/anomaly_labels.csv` and this cell turns the section into a
   real result.

The template below writes a stratified sample to label, so the labelling effort is spent on
the rows that matter rather than on 200 random ordinary listings.

In [ ]:
from laptop_price.paths import ANOMALY_LABELS, LABELS_DIR

if ANOMALY_LABELS.exists():
    labels = pd.read_csv(ANOMALY_LABELS).set_index("listing_index")["label"]
    from laptop_price.anomaly.deals import precision_at_k
    for k in (10, 25, 50):
        p = precision_at_k(deals, labels, k=k)
        print(f"precision@{k}: {p:.2%}" if p == p else f"precision@{k}: no labelled rows in the top {k}")
else:
    LABELS_DIR.mkdir(parents=True, exist_ok=True)
    template_path = LABELS_DIR / "anomaly_labels_TEMPLATE.csv"
    sample = pd.concat([
        deals.head(60).assign(sampled_from="top_deals"),
        matrix[scores["is_outlier"]].head(60).assign(sampled_from="joint_outlier"),
        matrix.sample(80, random_state=42).assign(sampled_from="random"),
    ])
    out = sample[[TARGET, "RAM_SIZE", "SSD_SIZE", "cpu_mark", "gpu_g3d_mark",
                  "brand", "city_grouped", "sampled_from"]].copy()
    out.index.name = "listing_index"
    out["label"] = ""      # fill in with: scam | normal | bargain
    out.to_csv(template_path)
    print(f"No labels yet. Wrote a {len(out)}-row labelling template to:\n  {template_path}")
    print("Fill the 'label' column, save it as anomaly_labels.csv, and re-run this cell.")

## Summary

- **Scam detection** flags the joint (specs, price) outliers that no single-column filter
  would catch.
- **The deal finder** is the most valuable output in the project and cost almost nothing on
  top of the existing regression model.
- **Spec-inconsistency** ties the association rules into the anomaly work, so the three
  deliverables form one argument.
- **Evaluation** is agreement-based today and becomes precision@k the moment ~200 listings
  are hand-labelled.

### The honest caveat

Every flag here is relative to a model of *asking* prices. A listing far below the
estimate might be a bargain, a scam, a damaged machine, an urgent seller, or a spec the
scraper misread. The detectors narrow the candidates; they do not adjudicate them.